# Benchmark CLIP vs SigLIP trên MS COCO (Karpathy test)

Notebook này chạy **toàn bộ pipeline** trên Kaggle GPU:

1. Clone repo.
2. Tải annotations Karpathy (`dataset_coco.json`) — tự verify đúng 5.000 ảnh test / 25.000 captions.
3. Tải trực tiếp **5.000 ảnh test** từ COCO (~883 MB, ~9 phút) — **không cần Add Data**.
4. Verify đủ 5000 ảnh trước khi chạy.
5. Chạy benchmark CLIP và SigLIP, lưu kết quả JSON.

**Chuẩn bị:**

- *Settings → Accelerator → GPU T4 x2* (hoặc P100)
- *Settings → Internet → **On*** (bắt buộc — cần cho git clone, tải annotations, tải ảnh, tải model)
- **Không cần Add Data.** Ảnh tải trực tiếp từ `images.cocodataset.org` và được đối chiếu
  SHA256 với 5 ảnh mẫu trong repo để chắc chắn là bản gốc (không bị resize).

Sau đó **Run All** (lần đầu nên chạy `--limit 20` ở cell benchmark để thử trước).

Kết quả cuối: `results/benchmark.json` (nằm trong `/kaggle/working` nên được lưu vào output).

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nghon4maeri/clip-siglip-semantic-retrieval.git"
BRANCH = "setup/standardize-benchmark"  # đổi thành tên branch nếu cần test trước khi merge
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "clip-siglip-semantic-retrieval"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("Repo  :", REPO_DIR)
print("Commit:", commit)

In [ ]:
import transformers

major, minor = map(int, transformers.__version__.split(".")[:2])
if (major, minor) < (4, 36):
    print(f"transformers {transformers.__version__} quá cũ, đang nâng cấp...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.36"], check=True)
else:
    print(f"transformers {transformers.__version__} — OK (cần >= 4.36 cho SigLIP)")

In [ ]:
!python datasets/download_annotations.py

### (Tùy chọn) Nếu bạn đã Add Data dataset COCO có `val2014`

Cell dưới đây symlink thư mục `val2014` từ `/kaggle/input/` vào repo, để **bỏ qua bước tải ảnh**
(nhanh hơn ~9 phút). Nếu không dùng thì **bỏ qua cell này** — cell tải ảnh ngay sau sẽ tự lo.

> ⚠️ Dataset trên Kaggle do người khác upload, **có thể đã bị resize** (ví dụ dataset
> `rmfkxhxh/mscoco2014-resized` chỉ 1.4 GB so với bản gốc 6.6 GB) → số Recall sẽ **sai**.
> Vì vậy cell verify phía dưới có đối chiếu SHA256 5 ảnh mẫu: nếu báo `[LỖI]` thì đừng dùng
> nguồn đó, hãy quay lại cách tải trực tiếp.

In [ ]:
# (Tùy chọn) Dùng ảnh từ /kaggle/input thay vì tải. Bỏ qua cell này nếu không Add Data.
import glob
from pathlib import Path

target = Path("datasets/coco/images/val2014")
candidates = set()
for pattern in ("/kaggle/input/*/val2014", "/kaggle/input/*/*/val2014", "/kaggle/input/*/*/*/val2014"):
    candidates.update(glob.glob(pattern))


def has_coco_images(path):
    try:
        return any(f.startswith("COCO_val2014") and f.endswith(".jpg") for f in os.listdir(path)[:100])
    except OSError:
        return False


candidates = sorted(p for p in candidates if has_coco_images(p))
if not candidates:
    print("Không tìm thấy val2014 trong /kaggle/input — bỏ qua, cell tải ảnh sẽ lo.")
else:
    image_source = Path(candidates[0])
    target.parent.mkdir(parents=True, exist_ok=True)
    for p in target.parent.glob("val2014*"):
        if p.is_symlink():
            p.unlink()
    target.symlink_to(image_source)
    n = sum(1 for f in os.listdir(image_source) if f.endswith(".jpg"))
    print(f"Nguồn     : {image_source} ({n:,} file .jpg)")
    print(f"Symlink   : {target} -> {image_source}")
    print("Lưu ý: cell tải ảnh ngay sau sẽ tự bỏ qua nếu đủ 5000 ảnh đã có.")

In [ ]:
# Tải trực tiếp 5.000 ảnh test từ COCO (~883 MB, ~9 phút) — không cần Add Data.
# Ảnh là bản gốc: script đối chiếu SHA256 với 5 ảnh mẫu trong repo.
# Idempotent: chạy lại sẽ bỏ qua ảnh đã có (kể cả ảnh từ /kaggle/input nếu đã symlink).
!python scripts/download_test_images.py --workers 16

In [ ]:
!python scripts/verify_data.py

In [ ]:
# Chạy benchmark: mặc định cả 2 model với checkpoint B/16, protocol 5 captions/ảnh.
# Thêm --limit N nếu chỉ muốn chạy thử.
!python src/evaluation/benchmark.py --models clip,siglip --batch-size 64 --output results/benchmark.json

In [ ]:
import json

with open("results/benchmark.json", encoding="utf-8") as f:
    res = json.load(f)

print(f"git commit       : {res['git_commit']}")
print(f"annotations sha  : {res['annotation_sha256']}")
print(f"device           : {res['device']}")
print(f"dataset          : {res['dataset']}")
print()
header = f"{'Model':<8} {'i2t R@1':>8} {'i2t R@5':>8} {'i2t R@10':>9} {'t2i R@1':>8} {'t2i R@5':>8} {'t2i R@10':>9} {'Time(s)':>8} {'VRAM(MB)':>9}"
print(header)
print("-" * len(header))
for name, r in res["results"].items():
    m = r["metrics"]
    print(f"{name:<8} {m['i2t_R@1']:>8.2f} {m['i2t_R@5']:>8.2f} {m['i2t_R@10']:>9.2f} "
          f"{m['t2i_R@1']:>8.2f} {m['t2i_R@5']:>8.2f} {m['t2i_R@10']:>9.2f} "
          f"{r['timing']['total_seconds']:>8.1f} {str(r['peak_vram_mb']):>9}")

### Ghi chú

- Đây là **inference/benchmark** trên model pretrained, **không train**. Kaggle chỉ cung cấp GPU để chạy đánh giá.
- Số liệu chỉ có giá trị khi chạy đủ 5000 ảnh (không dùng `--limit`) và với đúng file annotations đã verify.
- **Đối chiếu tham khảo:** paper SigLIP (Zhai et al., ICCV 2023) Table 3 báo cáo trên cùng cấu hình
  ViT-B/16 @ 224px — CLIP i2t R@1 ≈ 52.4 / t2i R@1 ≈ 33.1; SigLIP i2t R@1 ≈ 64.4 / t2i R@1 ≈ 47.2.
  Lệch 1–3 điểm là bình thường (khác prompt/pipeline); lệch >10 điểm là có lỗi.
- Đổi checkpoint: thêm `--clip-checkpoint <hf-id>` / `--siglip-checkpoint <hf-id>`.
- File kết quả `results/benchmark.json` được lưu cùng output của notebook.